# Sprint 1 - AI Application Anatomy, Model Calls, and Structured Outputs

This notebook follows LS1-LS4. You will map what the application owns, compare model access paths, run one direct OpenRouter call, then turn model text into validated JSON through the shared helper core.


## 1. Install the helper core from GitHub

Run this first in Colab. It installs the current shared helper core directly from GitHub before any helper imports.


In [ ]:
#@title Install helper core from GitHub
%pip install -q --force-reinstall --no-cache-dir "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"

print("Installed helper core from GitHub main.")

## 2. Setup checkpoint

The helper core setup is handled before the live walkthrough. In a prepared Colab copy, use the next code cells to check whether `OPENROUTER_API_KEY` is available, import the helper modules, and print the course-enabled models.

Configure `OPENROUTER_API_KEY` before running this notebook. The chat, structured-output, and LangGraph examples make real model calls and consume API credits.


In [ ]:
from notebook_setup import require_openrouter_key

_ = require_openrouter_key(prompt=True)

In [ ]:
from pydantic import BaseModel, Field

from models import ChatModel, all_allowed_model_ids
from openrouter import OpenRouterClient
from structured_graph import StructuredOutputGraph

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-1")
print("Course-enabled models:")
for model_id in all_allowed_model_ids():
    print("-", model_id)

## 3. Map the application boundary

LS1 asks you to trace a request, not guess from the interface. Keep the model in the middle and name the responsibilities around it.


In [ ]:
APPLICATION_COMPONENTS = [
    {
        "component": "Input",
        "app_owns": "Collect the user's message and any product state that matters.",
    },
    {
        "component": "Retrieval",
        "app_owns": "Choose evidence, rank it, and decide what reaches the prompt.",
    },
    {
        "component": "Model access",
        "app_owns": "Select an enabled model, build messages, and send the request safely.",
    },
    {
        "component": "Tools",
        "app_owns": "Validate tool calls, run application code, and handle failures.",
    },
    {
        "component": "Output contract",
        "app_owns": "Validate the model result before another component uses it.",
    },
]

for item in APPLICATION_COMPONENTS:
    print(f"{item['component']}: {item['app_owns']}")

## 4. Compare the three access paths

LS2 frames hosted UI, embedded copilot, and direct API as different ownership choices. The notebook path uses direct API access because the application needs a testable boundary.


In [ ]:
MODEL_ACCESS_PATHS = [
    {
        "path": "Hosted UI",
        "best_when": "A person is exploring or drafting.",
        "app_control": "Low: the product does not own the request path.",
    },
    {
        "path": "Embedded copilot",
        "best_when": "The workflow already lives inside another product.",
        "app_control": "Medium: the host product controls much of the boundary.",
    },
    {
        "path": "Direct API",
        "best_when": "Software needs repeatable behavior, logging, validation, and tests.",
        "app_control": "High: the app owns messages, model choice, and downstream handling.",
    },
]

for route in MODEL_ACCESS_PATHS:
    print(f"{route['path']}: {route['best_when']} {route['app_control']}")

## 5. Make a direct LLM call

The direct call shows the provider boundary: messages go out, a provider response comes back, and the application decides what to do next.


In [ ]:
require_openrouter_key(prompt=True)

messages = [
    {
        "role": "system",
        "content": "You are a concise AI app development teaching assistant.",
    },
    {
        "role": "user",
        "content": "In three bullets, explain what an AI application owns around the model.",
    },
]

response = client.chat(
    messages,
    model=ChatModel.GEMINI_25_FLASH_LITE,
    temperature=0.2,
    max_tokens=300,
)
print("Model returned text:")
print(response.content)
print("\nProvider model:", response.model)
print("Finish reason:", response.finish_reason)

## 6. Move from text to validated JSON

LS3 moves from an answer a human can read to data the app can safely use. Prompting asks for a shape; validation decides whether the result may cross the application boundary.


In [ ]:
require_openrouter_key(prompt=True)


class LessonRoute(BaseModel):
    task_type: str = Field(
        description="One of: chat, structured_output, retrieval, tools."
    )
    needs_retrieval: bool = Field(
        description="Whether the next step should search course material."
    )
    confidence: float = Field(ge=0, le=1, description="Confidence in the route.")
    next_action: str = Field(description="A short instruction for the app scaffold.")


route = client.structured(
    [
        {
            "role": "system",
            "content": "Route student requests for an AI app development notebook.",
        },
        {
            "role": "user",
            "content": "I need examples of why naive RAG retrieves irrelevant chunks.",
        },
    ],
    output_model=LessonRoute,
    schema_name="lesson_route",
    model=ChatModel.GEMINI_31_FLASH_LITE,
    temperature=0.1,
)

print(route.model_dump_json(indent=2))

## 7. Wrap the same idea in LangGraph

LangGraph makes the steps inspectable: prepare messages, call the model, and return a typed object. This mirrors the scaffold habit from LS4: prove the path, then defend the extension.


In [ ]:
require_openrouter_key(prompt=True)


class LabPlan(BaseModel):
    objective: str
    steps: list[str] = Field(min_length=2, max_length=5)
    success_check: str


graph = StructuredOutputGraph(
    client=client,
    output_model=LabPlan,
    schema_name="lab_plan",
    system_prompt="Create compact campus lab plans for AI app development students.",
    model=ChatModel.GEMINI_31_FLASH_LITE,
)

plan = graph.invoke(
    "Plan a 20 minute activity where students compare text output and JSON output."
)
print(plan.model_dump_json(indent=2))

## 8. Sprint 1 checkpoint defense

Use this checklist to connect the notebook back to LS4. A strong checkpoint answer names what the application controls and points to evidence in the run.


In [ ]:
checkpoint_evidence = [
    ("Application responsibility", "APPLICATION_COMPONENTS names what the app owns."),
    (
        "Access path",
        "MODEL_ACCESS_PATHS explains why this notebook uses direct API access.",
    ),
    ("Provider boundary", "client.chat sends messages and returns a response object."),
    ("Structured contract", "LessonRoute rejects data that does not match the schema."),
    (
        "Extension opportunity",
        "Retrieval and tools are visible next steps, not hidden prompt tricks.",
    ),
]

for claim, evidence in checkpoint_evidence:
    print(f"{claim}: {evidence}")

## 9. FieldCare project checkpoint: application contract

Start the module project here rather than waiting for Sprint 4. Define the narrow FieldCare request families your first version will handle, the inputs it needs, the response shape it promises, and the routing rules that keep unsafe or unsupported requests out of the normal answer path.

Edit the contract to match decisions you can defend from this sprint. The validator checks completeness and unresolved placeholders; it does not choose the scope for you. Download the JSON when the cell passes and keep it for Sprint 4.


In [ ]:
from fieldcare import write_fieldcare_artifact

FIELDCARE_APP_CONTRACT = {
    "schema_version": "1.0",
    "project_id": "fieldcare",
    "supported_request_types": [
        "troubleshooting_plus_warranty",
        "safety_escalation",
        "ticket_status_only",
    ],
    "out_of_scope_request_types": [
        "incomplete_request",
        "unsupported_model",
        "unsupported_business_promise",
    ],
    "required_inputs": ["request_text", "equipment_id", "ticket_id"],
    "response_contract": {
        "required_fields": [
            "answer_summary",
            "next_checks",
            "coverage_statement",
            "evidence",
            "tool_state",
            "decision_flags",
            "escalation_path",
            "limitations",
        ]
    },
    "routing_rules": [
        {"when": "safety_indicator", "route_to": "Safety Engineering"},
        {"when": "repeat_fault", "route_to": "Field Engineering"},
        {"when": "warranty_unavailable", "route_to": "Warranty Operations"},
    ],
    "success_criteria": [
        "Every factual claim is traceable to a current document or tool result.",
        "Coverage claims use current warranty state rather than documentation alone.",
        "Unsafe, unsupported, or underspecified requests abstain, ask, or escalate.",
    ],
}

APP_CONTRACT_PATH = write_fieldcare_artifact(
    FIELDCARE_APP_CONTRACT,
    "app_contract",
    "fieldcare_app_contract.json",
)
print(f"Validated project contract: {APP_CONTRACT_PATH}")

In [ ]:
#@title Download the Sprint 1 project artifact { display-mode: "form" }
try:
    from google.colab import files

    files.download(str(APP_CONTRACT_PATH))
except ImportError:
    print(APP_CONTRACT_PATH)